# PHASE 3: DATA CLEANING & WRANGLING

The objective of this phase is to prepare the each of the 3 datasets for analysis by ensuring it is clean, consistent, and structured appropriately. 
By the end of this phase, our datasets should be **clean**, **uniform**, and **ready** for analysis.

In [4]:
#Inspect column structure and data type

import pandas as pd

# Load samples from each dataset
dot1_sample = pd.read_csv("merged_datasets/dot1_merged.csv", nrows=10)
dot2_sample = pd.read_csv("merged_datasets/dot2_merged.csv", nrows=10)
dot3_sample = pd.read_csv("merged_datasets/dot3_merged.csv", nrows=10)

# Display column names and datatypes
print("DOT1 Columns:\n", dot1_sample.dtypes)
print("\nDOT2 Columns:\n", dot2_sample.dtypes)
print("\nDOT3 Columns:\n", dot3_sample.dtypes)


DOT1 Columns:
 TRDTYPE             int64
USASTATE           object
DEPE               object
DISAGMOT            int64
MEXSTATE           object
CANPROV            object
COUNTRY             int64
VALUE               int64
SHIPWT              int64
FREIGHT_CHARGES     int64
DF                  int64
CONTCODE           object
MONTH               int64
YEAR                int64
SOURCE_FILE        object
dtype: object

DOT2 Columns:
 TRDTYPE             int64
USASTATE           object
COMMODITY2          int64
DISAGMOT            int64
MEXSTATE           object
CANPROV            object
COUNTRY             int64
VALUE               int64
SHIPWT              int64
FREIGHT_CHARGES     int64
DF                  int64
CONTCODE           object
MONTH               int64
YEAR                int64
SOURCE_FILE        object
dtype: object

DOT3 Columns:
 TRDTYPE             int64
DEPE                int64
COMMODITY2          int64
DISAGMOT            int64
COUNTRY             int64
VALUE          

In [7]:
#Align column structure across the 3 dataset,fill categorical columns with 'Unknown', leave numerical columns as NaN  
    

import pandas as pd
import numpy as np

# Load the individual datasets (already saved earlier)
dot1 = pd.read_csv("merged_datasets/dot1_merged.csv", dtype=str)
dot2 = pd.read_csv("merged_datasets/dot2_merged.csv", dtype=str)
dot3 = pd.read_csv("merged_datasets/dot3_merged.csv", dtype=str)

# Convert column names to uppercase and strip whitespace
dot1.columns = dot1.columns.str.upper().str.strip()
dot2.columns = dot2.columns.str.upper().str.strip()
dot3.columns = dot3.columns.str.upper().str.strip()

# Get the union of all columns
all_columns = sorted(set(dot1.columns) | set(dot2.columns) | set(dot3.columns))

# Columns to treat as categorical (based on your structure)
categorical_cols = [
    'USASTATE', 'MEXSTATE', 'CANPROV', 'COUNTRY', 'DISAGMOT',
    'CONTCODE', 'SOURCE_FILE', 'TRDTYPE', 'COMMODITY2'
]

# Function to align and fill
def align_and_fill(df, name):
    # Add missing columns
    for col in all_columns:
        if col not in df.columns:
            df[col] = pd.NA

    # Ensure consistent column order
    df = df[all_columns]

    # Fill categorical with 'Unknown'
    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    # Leave numerical as NaN (do nothing)

    # Save aligned output
    output_file = f"aligned_{name}.csv"
    df.to_csv(output_file, index=False)
    print(f" Saved aligned file: {output_file}")
    return df

# Align and save all
dot1_aligned = align_and_fill(dot1, 'dot1')
dot2_aligned = align_and_fill(dot2, 'dot2')
dot3_aligned = align_and_fill(dot3, 'dot3')



 Saved aligned file: aligned_dot1.csv
 Saved aligned file: aligned_dot2.csv
 Saved aligned file: aligned_dot3.csv


In [8]:
# Clean and Normalize Numeric Columns across 3 datasets
import pandas as pd
import numpy as np

# Files to process
files = ['aligned_dot1.csv', 'aligned_dot2.csv', 'aligned_dot3.csv']
output_files = ['cleaned_dot1.csv', 'cleaned_dot2.csv', 'cleaned_dot3.csv']

# Numeric columns to clean
numeric_cols = ['VALUE', 'SHIPWT', 'FREIGHT_CHARGES', 'DF', 'MONTH', 'YEAR', 'DEPE']

for file, output in zip(files, output_files):
    print(f" Cleaning numeric columns in {file}...")
    chunk_iter = pd.read_csv(file, dtype=str, chunksize=500_000)
    
    with open(output, 'w', encoding='utf-8', newline='') as f_out:
        first = True
        for chunk in chunk_iter:
            for col in numeric_cols:
                if col in chunk.columns:
                    # Remove commas, spaces, and convert to numeric
                    chunk[col] = (
                        chunk[col].str.replace(',', '', regex=False)
                                   .str.strip()
                                   .replace('', np.nan)
                    )
                    chunk[col] = pd.to_numeric(chunk[col], errors='coerce')
            
            # Write to cleaned file
            chunk.to_csv(f_out, index=False, header=first, mode='a')
            first = False

    print(f" Saved cleaned file: {output}")


🔍 Cleaning numeric columns in aligned_dot1.csv...
 Saved cleaned file: cleaned_dot1.csv
🔍 Cleaning numeric columns in aligned_dot2.csv...
 Saved cleaned file: cleaned_dot2.csv
🔍 Cleaning numeric columns in aligned_dot3.csv...
 Saved cleaned file: cleaned_dot3.csv


In [ ]:
# Check for duplicate rows and eliminate them

#removing duplicates for dot 1
import pandas as pd

files = ['cleaned_dot1.csv']
output_files = ['deduped_dot1.csv']
chunk_size = 500_000

for file, output in zip(files, output_files):
    print(f"\nDeduplicating {file}...")

    seen = set()
    header_written = False

    for chunk in pd.read_csv(file, chunksize=chunk_size, dtype=str):
        # Convert rows to hashable tuples for fast lookup
        row_tuples = chunk.astype(str).apply(lambda row: tuple(row), axis=1)
        is_duplicate = row_tuples.isin(seen)

        # Keep only unique rows
        unique_chunk = chunk[~is_duplicate]

        # Update seen
        seen.update(row_tuples[~is_duplicate])

        # Save in append mode
        unique_chunk.to_csv(output, mode='a', header=not header_written, index=False)
        header_written = True

        print(f"Saved {len(unique_chunk)} new unique rows")

    print(f"Done with {file} → {output}")



Deduplicating cleaned_dot1.csv...
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 377214 new unique rows
Saved 405119 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 211492 new unique rows
 Done with cleaned_dot1.csv → deduped_dot1.csv

Deduplicating cleaned_dot2.csv...
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 new unique rows
Saved 500000 

: 

In [ ]:
#removing duplicates for dot3
import pandas as pd
import os

file = 'cleaned_dot3.csv'         # Change to dot3 as needed
output = 'deduped_dot3.csv'
temp_file = 'temp_partial_dedup.csv'
chunk_size = 500_000

# Remove existing temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

# First pass: deduplicate within each chunk
for chunk in pd.read_csv(file, chunksize=chunk_size, dtype=str):
    chunk = chunk.drop_duplicates()
    chunk.to_csv(temp_file, mode='a', index=False, header=not os.path.exists(temp_file))
    print(f"Wrote chunk to temp file: {len(chunk)} rows")

# Second pass: full deduplication
print("Performing final deduplication...")
df = pd.read_csv(temp_file, dtype=str)
df.drop_duplicates().to_csv(output, index=False)
print(f" Done: {output} saved. Shape: {df.shape}")


Wrote chunk to temp file: 499986 rows
Wrote chunk to temp file: 499980 rows
Wrote chunk to temp file: 499980 rows
Wrote chunk to temp file: 499983 rows
Wrote chunk to temp file: 499982 rows
Wrote chunk to temp file: 499982 rows
Wrote chunk to temp file: 499977 rows
Wrote chunk to temp file: 499979 rows
Wrote chunk to temp file: 499982 rows
Wrote chunk to temp file: 499988 rows
Wrote chunk to temp file: 133290 rows
Performing final deduplication...
 Done: deduped_dot3.csv saved. Shape: (5133109, 16)


In [2]:
import pandas as pd
import os
import hashlib

file = 'cleaned_dot2.csv'
output = 'deduped_dot2.csv'
temp_file = 'temp_partial_dedup.csv'
chunk_size = 500_000

# Reset temp file
if os.path.exists(temp_file):
    os.remove(temp_file)

seen_hashes = set()

def hash_row(row):
    return hashlib.md5(pd.util.hash_pandas_object(row, index=False).values).hexdigest()

# First pass – drop based on row hashes
for chunk in pd.read_csv(file, chunksize=chunk_size, dtype=str):
    chunk['__rowhash__'] = chunk.apply(hash_row, axis=1)
    chunk = chunk[~chunk['__rowhash__'].isin(seen_hashes)]
    seen_hashes.update(chunk['__rowhash__'])
    chunk.drop(columns='__rowhash__', inplace=True)
    chunk.to_csv(temp_file, mode='a', index=False, header=not os.path.exists(temp_file))
    print(f"Wrote chunk – temp size: {os.path.getsize(temp_file)/1e6:.2f} MB")

# Final deduplication (lightweight)
df = pd.read_csv(temp_file, dtype=str)
df.to_csv(output, index=False)
print(f"Deduplicated and saved to {output} | Final shape: {df.shape}")


Wrote chunk – temp size: 35.26 MB
Wrote chunk – temp size: 71.00 MB
Wrote chunk – temp size: 106.00 MB
Wrote chunk – temp size: 141.53 MB
Wrote chunk – temp size: 177.04 MB
Wrote chunk – temp size: 212.59 MB
Wrote chunk – temp size: 248.10 MB
Wrote chunk – temp size: 283.58 MB
Wrote chunk – temp size: 318.71 MB
Wrote chunk – temp size: 354.19 MB
Wrote chunk – temp size: 389.70 MB
Wrote chunk – temp size: 425.58 MB
Wrote chunk – temp size: 461.27 MB
Wrote chunk – temp size: 496.78 MB
Wrote chunk – temp size: 532.64 MB
Wrote chunk – temp size: 568.34 MB
Wrote chunk – temp size: 604.19 MB
Wrote chunk – temp size: 639.72 MB
Wrote chunk – temp size: 660.38 MB
Wrote chunk – temp size: 668.06 MB
Wrote chunk – temp size: 703.55 MB
Wrote chunk – temp size: 738.95 MB
Wrote chunk – temp size: 774.48 MB
Wrote chunk – temp size: 810.04 MB
Wrote chunk – temp size: 845.65 MB
Wrote chunk – temp size: 881.53 MB
Wrote chunk – temp size: 917.12 MB
Wrote chunk – temp size: 952.56 MB
Wrote chunk – temp siz